In [14]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [15]:
df = pd.read_csv('../data/raw/shopping_trends.csv')

This notebook goes through some useful data cleaning techniques. Take note that the exploratory process has already confirmed for us that there are no missing or duplicate values in this dataset. The outputs for handling missing or duplicate values will therefore not change anything for this particular dataset. In other cases however, rows with missing values can either be removed from the dataset or the missing values can be imputed using various imputation strategies.

Imputation strategies,
1. Mean/Median/Mode: replacing missing values with the column mean, median or mode
2. Constant/Fixed value imputation: replacing missing values with a placeholder like '0' ot the word "Unknown"
3. K-Nearest Neighbour (KNN): replacing missing values by finding the closest matching rows based on data from other variables
4. Regression imputation: using a regression model built from the data available in other columns to estimate missing data points
5. Forward/Backward fill: using the last known value to replace the subsequent or preceding missing value. This technique is particularly useful for time-series data.

Determining which imputation strategy is the most appropriate requires a good understanding of the relationships between variables in the dataset. Say for example, there are missing values in the 'Discount Applied' column of the dataset. The customer gets a discount if they used a promo code as well as a credit card as payment method. As a practitioner, 
you could write a for loop which checks these two conditions and uses that information to impute the missing values in the 'Discount Applied' column, use KNN imputation which uses the information in the 'Payment Method' and 'Promo Code' columns to impute the missing values in the 'Discount Applied' column.

An appropriate way to handle missing values in the 'Age' column would be to simply remove the row especially if the age variable is one of the main variables of interest for your analysis. Using an imputation method for this variable may result in the wrong conclusions at the end of the analysis.

Mean/Median/Mode Imputation:

In [16]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')
numeric_columns = df.drop(columns=["Customer ID"], errors='ignore').select_dtypes(include=np.number).columns
df_imputed = df.copy()
df_imputed[numeric_columns] = imputer.fit_transform(df[numeric_columns])

In [17]:
imputer = SimpleImputer(strategy='median')
numeric_columns = df.drop(columns=["Customer ID"], errors='ignore').select_dtypes(include=np.number).columns
df_imputed = df.copy()
df_imputed[numeric_columns] = imputer.fit_transform(df[numeric_columns])

In [18]:
imputer = SimpleImputer(strategy='most_frequent')
numeric_columns = df.drop(columns=["Customer ID"], errors='ignore').select_dtypes(include=np.number).columns
df_imputed = df.copy()
df_imputed[numeric_columns] = imputer.fit_transform(df[numeric_columns])

Constant/Fixed value Imputation:

In [19]:
imputer = SimpleImputer(strategy='constant', fill_value=0) #or imputer = SimpleImputer(strategy='constant', fill_value='Unknown') for categorical columns
numeric_columns = df.drop(columns=["Customer ID"], errors='ignore').select_dtypes(include=np.number).columns
df_imputed = df.copy()
df_imputed[numeric_columns] = imputer.fit_transform(df[numeric_columns])

K-Nearest Neighbour (KNN) Imputation:

In [20]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=2, weights="distance") #or weights="uniform" if you want to give equal weight to all neighbors
numeric_columns = df.drop(columns=["Customer ID"], errors='ignore').select_dtypes(include=np.number).columns
imputed_data = imputer.fit_transform(df[numeric_columns])
df_imputed = pd.DataFrame(imputed_data, columns=df[numeric_columns].columns)

Regression Imputation:

In [22]:
from sklearn.experimental import enable_iterative_imputer  
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
numeric_columns = df.drop(columns=["Customer ID"], errors='ignore').select_dtypes(include=np.number).columns
imputer = IterativeImputer(estimator=lr, max_iter=10, random_state=0)
imputed_array = imputer.fit_transform(df[numeric_columns])
df_imputed = pd.DataFrame(imputed_array, columns=df[numeric_columns].columns)

Other data cleaning processes include but are not limited to, fixing structural errors, standardizing data formats, handling outliers and correcting data types.

To find structural errors for example inconsistent naming conventions in the dataset, one needs to check both the column names and all columns of datatype 'str'

In [3]:
def check_text_inconsistencies(dataframe, column_name):
    print(f"\n Analyzing values in column: '{column_name}'...")
    raw_counts = dataframe[column_name].value_counts()
    standardized = dataframe[column_name].astype(str).str.strip().str.lower()
    for clean_val in standardized.unique():
        variants = dataframe[standardized == clean_val][column_name].unique()
        if len(variants) > 1:
            print(f"Found {len(variants)} naming variants for root value '{clean_val}':")
            for v in variants:
                print(f"   - '{v}' (occurs {raw_counts[v]} times)")
                
categorical_columns = df.select_dtypes(include=["object", "string"]).columns
for column in categorical_columns:
    check_text_inconsistencies(df, df[column].name)


 Analyzing values in column: 'Gender'...

 Analyzing values in column: 'Item Purchased'...

 Analyzing values in column: 'Category'...

 Analyzing values in column: 'Location'...

 Analyzing values in column: 'Size'...

 Analyzing values in column: 'Color'...

 Analyzing values in column: 'Season'...

 Analyzing values in column: 'Subscription Status'...

 Analyzing values in column: 'Payment Method'...

 Analyzing values in column: 'Shipping Type'...

 Analyzing values in column: 'Discount Applied'...

 Analyzing values in column: 'Promo Code Used'...

 Analyzing values in column: 'Preferred Payment Method'...

 Analyzing values in column: 'Frequency of Purchases'...


When conducting an exploratory data analysis project, it is important to allow the data to guide your decision-making rather than relying on predetermined assumptions. Decisions regarding the treatment of outliers, the choice of statistical methods, and any data transformations should be driven by the evidence provided by the data and informed by the context of the problem being investigated.

There is no single correct approach to handling outliers. Instead, the most appropriate method depends on the characteristics of the dataset and the objectives of the analysis. A good starting point is to develop a thorough understanding of the relationships between variables and their underlying distributions which is done through the EDA process.

From there, the next step is to define a clear research question or analytical objective. Once this has been established, you can select the statistical or analytical techniques that are most appropriate for addressing the question, taking into account the patterns, distributions, and potential outliers identified during the exploratory phase.

Strategies for handling outliers include,

1. trimming extreme values 
2. transforming extreme values
3. using imputation techniques to replace extreme values 
4. selecting analytical techniques that are robust against outliers 

The following code blocks depict the four outlier handling techniques discussed above. 

Reminder that this particular dataset does not have any outlier values thus ultimately no changes in the dataset will occur.

Trimming using the IQR method

In [23]:
numeric_columns = df.drop(columns=["Customer ID"], errors='ignore').select_dtypes(include=np.number).columns
Q1 = df[numeric_columns].quantile(0.25)
Q3 = df[numeric_columns].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df_trimmed = df[(df[numeric_columns] >= lower_bound) & (df[numeric_columns] <= upper_bound)]

Capping/Winsorising

In [26]:
from scipy.stats.mstats import winsorize
df_copy = df.copy()
numeric_columns = df.drop(columns=["Customer ID"], errors='ignore').select_dtypes(include=np.number).columns
for col in numeric_columns:
    df_copy[col] = winsorize(df_copy[col], limits=[0.1, 0.1]) #capping the lowest and highest 10% of the data

Capping/Winsorising can also be done in pandas

In [ ]:
df_copy = df.copy()
numeric_columns = df.drop(columns=["Customer ID"], errors='ignore').select_dtypes(include=np.number).columns
for col in numeric_columns:
    df_copy[col] = df[numeric_columns].clip(lower=lower_bound[col], upper=upper_bound[col])

Mathematical Transformations of outliers,

Log transform:
This data transformation technique cannot handle negative values. All values must be >=0. 
Is best used for right-skewed positive data. 

In [ ]:
numeric_columns = df.drop(columns=["Customer ID"], errors='ignore').select_dtypes(include=np.number).columns
log_df = df.copy()
for col in numeric_columns:
    mask = df[col] >= 0
    log_df.loc[mask, col] = np.log1p(df.loc[mask, col])

Square-root Transform:
This technique cannot handle negative values. All values must be >=0. 
Is best used for mildly right-skewed data. 

In [ ]:
numeric_columns = df.drop(columns=["Customer ID"], errors='ignore').select_dtypes(include=np.number).columns
sqrt_df = df.copy()
for col in numeric_columns:
    mask = df[col] >= 0
    sqrt_df.loc[mask, col] = np.sqrt(df.loc[mask, col])

Box-Cox Transform:
This technique cannot handle negative values. All values must be strictly >0. 
Is best used for right-skewed positive data

In [ ]:
from scipy.stats import boxcox
boxcox_df = df.copy()
numeric_columns = df.drop(columns=["Customer ID"], errors='ignore').select_dtypes(include=np.number).columns
for col in numeric_columns:
    mask = df[col] > 0
    if mask.sum() > 1:  
        transformed, _ = boxcox(df.loc[mask, col])
        boxcox_df.loc[mask, col] = transformed


Yeo-Johnson Transform:
This technique can handle 0 values and negative values.

In [ ]:
from sklearn.preprocessing import PowerTransformer
yeojohnson_df = df.copy()
numeric_columns = df.drop(columns=["Customer ID"], errors='ignore').select_dtypes(include=np.number).columns
pt = PowerTransformer(method="yeo-johnson", standardize=False)
yeojohnson_df[numeric_columns] = pt.fit_transform(df[numeric_columns])

Any one of the abovementioned imputation techniques can be used to replace extreme values within the dataset as well. It is sometimes inappropriate to remove or replace outlier values and in that case, extensive research needs to be conducted to identify analytical techniques which are robust to extreme values.